- 배포 모델 확정 실험 — 정리 데이터(201,616) · 레시피(eff128 · lr 4.8e-4) · **max_len 4096**
- 비교선: 앵커 `11_01` 0.8588(정리 test) ~ exp1 0.8683(정리 test 재계산, 구 레시피 8192) 사이
- 4096 선택 근거: 절단 문서 0.8% · 유실 토큰 1.60%(8192는 0.02%)이고 최악 시퀀스가 1/2이라 micro_batch를 2배로 써 훈련 시간을 단축할 수 있음.
- lr 안전: LR range test 전환점 512 8.84e-3 · 8192 7.13e-3 → 4096은 그 사이, 운영 lr이 15배 아래

In [ ]:
import sys
sys.path.insert(0, "src")  # /workspace/src

from patent_train import TrainingRunner, TrainConfig, probe_batches

In [ ]:
SEARCH = False   # True=fast-fail 짧은 런(2 epoch로 유도) / False=풀런(아래 epochs 사용)

cfg = TrainConfig(
    backbone="axenc",       # backbones.BACKBONES 키
    loss="focal",
    loss_params={"alpha": 0.25, "gamma": 2},
    max_len=4096,
    eff_batch=128,          # 배치 재현
    micro_batch=128,        
    eval_micro_batch=128,
    learning_rate=4.8e-4,   
    weight_decay=0.01,
    warmup_ratio=0.1,
    epochs=12,
    early_stop_epochs=2,    
    notebook_name="16_01_Model_4096.ipynb",   # wandb code saving
    tag="modernbert-patent-len4096-op",
    run_name="axenc_len4096_focal_op",
    repo_final="ingyoun/A.X-patent-len4096-op",
    out_path="/workspace/output/modernbert-len4096-op",
    search=SEARCH,
)

print("run_name:", cfg.run_name, "| epochs:", cfg.epochs, "| grad_accum:", cfg.grad_accum)

## 구성 — 데이터·모델

토크나이저+원본 로드 → `max_len` 절단(캐시) → 분류기 구성. 단계를 나눠 중간 점검·부분 재실행이 가능하다.

In [ ]:
runner = TrainingRunner(cfg)

In [ ]:
runner.load_data()        # 토크나이저 + 원본 데이터셋(prep 캐시 있으면 원본 생략)
runner.data.raw

In [ ]:
runner.prepare_data()     # max_len 절단 → prep 캐시
runner.data.dataset

In [ ]:
runner.load_model()

## OOM 확인

GPU/배치에서 안전한 `micro_batch` 상한을 실측

In [ ]:

probe_batches(
    runner.model.to("cuda"), runner.data.tokenizer.vocab_size, cfg.max_len,
    train_mb=(4, 8, 16, 32, 64, 128),
    eval_mb=(8, 16, 32, 64, 96, 128, 160, 192, 224),
)

## 훈련

In [ ]:
runner.build_trainer()

In [ ]:
runner.train()

## 평가 · 메트릭 저장 · push

모델 가중치는 로컬에 두지 않고 Hub로만 올린다(팟을 지우면 로컬 사본은 사라진다). 로컬 사본이 필요하면 `runner.save_model()`.

In [ ]:
test_metrics = runner.evaluate("test")
for k, v in test_metrics.items():
    print(f"{k}: {v}")

In [ ]:
runner.save_metrics()     # runner.metrics(split 전체) → {tag}_metrics.json

In [ ]:
runner.push_to_hub()

## val·test 로짓 덤프

`logits_{tag}_{split}.npy`를 `out_path` 상위(`/workspace/output/`)에 저장

In [ ]:
runner.predict_logits("val")
runner.predict_logits("test")